# Sprint 3 — Sesión teórica consolidada  
## Introducción a SQL y KPIs financieros en Google Colab con DuckDB y Parquet

> **Duración:** 70 minutos  
> **Herramienta principal:** Google Colab  
> **Motor SQL:** DuckDB en memoria  
> **Dataset:** tablas Parquet cargadas desde GitHub  
> **Prefijo de archivos:** `webinar03_`  
> **Caso:** librería online / bookstore finance

Esta sesión consolida las sesiones teóricas del Sprint 3. El flujo original usaba una base local y archivos Parquet en una carpeta del computador. En esta versión, las tablas se leen desde GitHub y se crean dentro de Google Colab para que todos trabajen con el mismo entorno.

URL base esperada para los archivos:

```text
https://github.com/ljpiere/tpdata_course/raw/refs/heads/main/new_DA/DA_tutor/datasets/webinar03_*.parquet
```


## Agenda de la sesión

| Momento | Duración | Propósito |
|---|---:|---|
| 1. SQL en contexto | 8 min | Entender qué es SQL y por qué se usa en analítica |
| 2. Entorno Colab + DuckDB | 10 min | Crear tablas SQL desde Parquet en GitHub |
| 3. Modelo relacional Bookstore | 8 min | Reconocer tablas, llaves y relaciones |
| 4. Modelo mental de una consulta SQL | 8 min | Entender `FROM`, `WHERE`, `GROUP BY`, `SELECT`, `ORDER BY` |
| 5. Consultas básicas | 10 min | Usar `SELECT`, `WHERE`, `ORDER BY`, `LIMIT` |
| 6. Agregaciones y joins | 12 min | Construir KPIs con `GROUP BY` y `JOIN` |
| 7. KPIs financieros | 10 min | Calcular revenue, cost, gross profit, margin y ROI |
| 8. Cierre | 4 min | Validar aprendizaje, takeaways y canales de ayuda |


## Objetivos de aprendizaje

Al finalizar la sesión, el estudiante podrá:

- Explicar qué es SQL y cómo se usa en analítica de datos.
- Diferenciar entre tabla, archivo Parquet, motor SQL y notebook.
- Cargar tablas desde GitHub en Google Colab.
- Consultar datos usando DuckDB.
- Validar conteos antes de analizar.
- Usar `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, `GROUP BY` y `JOIN`.
- Construir KPIs financieros básicos: revenue, cost, gross profit, margin % y ROI.
- Traducir un resultado SQL a una interpretación de negocio.


## 1) SQL en contexto: qué es y para qué sirve

SQL significa **Structured Query Language**. Es un lenguaje declarativo para consultar, transformar y resumir datos organizados en tablas.

La palabra clave es **declarativo**: no le decimos al motor cada paso físico. Le decimos qué resultado queremos obtener. El motor decide cómo leer, filtrar, unir y calcular internamente.

En analítica de datos, SQL sirve para responder preguntas como:

- ¿Cuántas órdenes se hicieron?
- ¿Qué categoría generó más revenue?
- ¿Qué segmento de clientes paga más?
- ¿Qué productos tienen mayor margen?
- ¿Qué campañas tienen mejor ROI?

Flujo de trabajo de esta clase:

```text
Parquet en GitHub → Google Colab → DuckDB → SQL → Resultado tabular → Interpretación de negocio
```


## 2) Setup técnico en Google Colab

Ejecuta la siguiente celda al inicio de la clase.

La celda hace cuatro cosas:

1. instala dependencias;
2. define la URL base del repositorio;
3. lee cada archivo `.parquet`;
4. crea tablas SQL dentro de DuckDB.

Las tablas creadas serán:

- `customers`
- `products`
- `orders`
- `order_items`
- `payments`
- `campaigns`


In [ ]:
# Setup de Google Colab para practicar SQL con DuckDB y Parquet desde GitHub

!pip -q install duckdb pandas pyarrow

import duckdb
import pandas as pd

BASE_URL = "https://github.com/ljpiere/tpdata_course/raw/refs/heads/main/new_DA/DA_tutor/datasets"

TABLE_FILES = {
    "customers": "sprint03_customers.parquet",
    "products": "sprint03_products.parquet",
    "orders": "sprint03_orders.parquet",
    "order_items": "sprint03_order_items.parquet",
    "payments": "sprint03_payments.parquet",
    "campaigns": "sprint03_campaigns.parquet"
}

con = duckdb.connect(database=":memory:")

for table_name, file_name in TABLE_FILES.items():
    url = f"{BASE_URL}/{file_name}"
    df = pd.read_parquet(url)
    con.register(f"{table_name}_df", df)
    con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM {table_name}_df")

def sql(query):
    """Ejecuta una consulta SQL en DuckDB y devuelve un DataFrame de pandas."""
    return con.execute(query).df()

print("Tablas cargadas en DuckDB:")
print(list(TABLE_FILES.keys()))


## 3) Verificación de carga

Antes de analizar, validamos que las tablas existan y que tengan filas.

Resultado esperado:

| tabla | filas |
|---|---:|
| customers | 200 |
| products | 80 |
| orders | 600 |
| order_items | 902 |
| payments | 600 |
| campaigns | 6 |

Esta validación evita analizar datos incompletos o mal cargados.


In [ ]:
sql("""
SELECT 'customers' AS tabla, COUNT(*) AS filas FROM customers
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'payments', COUNT(*) FROM payments
UNION ALL
SELECT 'campaigns', COUNT(*) FROM campaigns;
""")

## 4) Modelo relacional del dataset

El dataset representa una librería online.

| Tabla | Qué contiene | Llave principal o de unión |
|---|---|---|
| `customers` | Clientes | `customer_id` |
| `products` | Libros/productos | `product_id` |
| `orders` | Órdenes | `order_id`, `customer_id`, `promo_code` |
| `order_items` | Líneas de cada orden | `order_id`, `product_id` |
| `payments` | Pagos por orden | `order_id` |
| `campaigns` | Campañas promocionales | `promo_code` |

Modelo mental:

```text
customers ── customer_id ── orders ── order_id ── payments
                                  │
                                  ├── order_items ── product_id ── products
                                  │
                                  └── promo_code ── campaigns
```

Una consulta analítica normalmente necesita responder tres preguntas:

| Pregunta | Ejemplo |
|---|---|
| ¿Qué tabla tiene la métrica? | `payments.amount_paid`, `order_items.quantity` |
| ¿Qué tabla tiene el contexto? | `products.category`, `customers.segment` |
| ¿Qué llave permite unirlas? | `order_id`, `product_id`, `customer_id` |


## 5) Modelo mental de una consulta SQL

Aunque escribimos una consulta empezando por `SELECT`, el orden lógico es:

```text
FROM      → de dónde salen los datos
WHERE     → qué filas se conservan
GROUP BY  → cómo se agrupan las filas
HAVING    → qué grupos se conservan
SELECT    → qué columnas o cálculos se muestran
ORDER BY  → cómo se ordena el resultado
LIMIT     → cuántas filas se muestran
```

Ejemplo:

```sql
SELECT
    status,
    COUNT(*) AS n_orders
FROM orders
WHERE order_date >= '2025-01-01'
GROUP BY status
ORDER BY n_orders DESC;
```

Lectura humana: toma la tabla `orders`, conserva órdenes desde 2025, agrupa por estado, cuenta órdenes y ordena el resultado de mayor a menor.


In [ ]:
sql("""
SELECT
    status,
    COUNT(*) AS n_orders
FROM orders
WHERE order_date >= '2025-01-01'
GROUP BY status
ORDER BY n_orders DESC;
""")

## 6) Consultas básicas: `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`

Primero exploramos una tabla sin unirla con otras.

Puntos clave:

- `SELECT *` sirve para explorar, no para reportes finales.
- `WHERE` filtra filas.
- `ORDER BY` ordena.
- `LIMIT` controla cuántas filas vemos.


In [ ]:
sql("""
SELECT
    product_id,
    title,
    category,
    price,
    cost,
    is_active
FROM products
WHERE is_active = TRUE
ORDER BY price DESC
LIMIT 10;
""")

## 7) Agregaciones: pasar de filas a KPIs

Un KPI resume muchas filas en una métrica.

Ejemplo: número de órdenes por estado.


In [ ]:
sql("""
SELECT
    status,
    COUNT(*) AS n_orders
FROM orders
GROUP BY status
ORDER BY n_orders DESC;
""")

## 8) Primer `JOIN`: unir contexto de cliente, orden y pago

Un `JOIN` no es solo “pegar tablas”. Es declarar una relación entre entidades usando llaves.

En este ejemplo:

- `orders` tiene la orden;
- `customers` agrega el segmento;
- `payments` agrega el valor pagado.


In [ ]:
sql("""
SELECT
    o.order_id,
    o.order_date,
    o.status,
    c.customer_name,
    c.segment,
    p.payment_method,
    p.amount_paid
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN payments p
    ON o.order_id = p.order_id
ORDER BY o.order_date DESC
LIMIT 20;
""")

## 9) KPI 1 — Revenue por categoría

Pregunta de negocio:

> ¿Qué categorías generan más revenue?

Fórmula:

```text
revenue = quantity * unit_price * (1 - discount_rate)
```

El revenue se calcula desde `order_items` porque allí están cantidad, precio unitario y descuento por línea. La categoría está en `products`, por eso necesitamos un `JOIN`.


In [ ]:
sql("""
SELECT
    pr.category,
    SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS revenue
FROM order_items oi
JOIN products pr
    ON oi.product_id = pr.product_id
GROUP BY pr.category
ORDER BY revenue DESC;
""")

## 10) KPI 2 — Cost, gross profit y margin %

Pregunta de negocio:

> ¿Qué categorías no solo venden más, sino que dejan mejor rentabilidad?

Fórmulas:

```text
cost = quantity * product_cost
gross_profit = revenue - cost
margin_pct = 100 * gross_profit / revenue
```

Regla técnica: usamos `NULLIF(revenue, 0)` para evitar división por cero.


In [ ]:
sql("""
SELECT
    pr.category,
    SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)) AS revenue,
    SUM(oi.quantity * pr.cost) AS cost,
    SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate))
        - SUM(oi.quantity * pr.cost) AS gross_profit,
    100 * (
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate))
        - SUM(oi.quantity * pr.cost)
    ) / NULLIF(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)), 0) AS margin_pct
FROM order_items oi
JOIN products pr
    ON oi.product_id = pr.product_id
GROUP BY pr.category
ORDER BY gross_profit DESC;
""")

## 11) KPI 3 — ROI por campaña

Pregunta de negocio:

> ¿Qué campañas promocionales devuelven más valor respecto a su inversión?

Fórmula:

```text
ROI % = 100 * (net_revenue - spend_usd) / spend_usd
```

Supuesto de clase: `amount_paid` representa revenue neto cobrado a nivel de orden. Para KPIs operativos, excluimos órdenes `cancelled` y `refunded`.


In [ ]:
sql("""
SELECT
    c.promo_code,
    c.campaign_name,
    c.channel,
    c.spend_usd,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(p.amount_paid) AS net_revenue,
    100 * (SUM(p.amount_paid) - c.spend_usd) / NULLIF(c.spend_usd, 0) AS roi_pct
FROM campaigns c
JOIN orders o
    ON c.promo_code = o.promo_code
JOIN payments p
    ON o.order_id = p.order_id
WHERE o.status NOT IN ('cancelled', 'refunded')
GROUP BY
    c.promo_code,
    c.campaign_name,
    c.channel,
    c.spend_usd
ORDER BY roi_pct DESC;
""")

## 12) Validación antes de reportar

Un resultado SQL no se reporta sin controles mínimos.

Checklist de validación:

- ¿Las tablas cargaron con las filas esperadas?
- ¿La llave de unión es correcta?
- ¿El `JOIN` multiplicó filas de forma no intencional?
- ¿El KPI debe excluir órdenes canceladas o reembolsadas?
- ¿La moneda y unidad están claras?
- ¿Las columnas calculadas tienen nombres interpretables?
- ¿El resultado responde una pregunta de negocio?

Plantilla de comunicación:

| Elemento | Ejemplo |
|---|---|
| Hallazgo | La categoría Data concentra el mayor gross profit. |
| Evidencia | Revenue, cost y margin calculados por categoría. |
| Implicación | Puede ser una categoría prioritaria para inventario o campañas. |
| Acción | Revisar disponibilidad y elasticidad de descuentos. |


## Preguntas de validación de conocimiento

Responde individualmente antes de cerrar.

| Pregunta | Respuesta esperada |
|---|---|
| ¿Qué diferencia hay entre un archivo Parquet y una tabla SQL en DuckDB? | El Parquet es el archivo de datos; DuckDB crea una tabla consultable a partir de ese archivo. |
| ¿Por qué validamos conteos antes de calcular KPIs? | Para confirmar que los datos cargaron completos y evitar conclusiones sobre datos incompletos. |
| ¿Qué tabla usarías para calcular revenue por línea? | `order_items`. |
| ¿Qué tabla usarías para traer la categoría del producto? | `products`. |
| ¿Qué llave une `orders` con `payments`? | `order_id`. |
| ¿Por qué se usa `GROUP BY`? | Para calcular métricas agregadas por una dimensión. |
| ¿Qué problema puede causar un `JOIN` incorrecto? | Duplicar filas e inflar métricas. |
| ¿Por qué usamos `NULLIF` en el margen? | Para evitar división por cero. |
| ¿Qué significa ROI en una campaña? | Retorno generado frente al gasto de campaña. |
| ¿Por qué SQL no termina en la consulta? | Porque el resultado debe interpretarse en lenguaje de negocio. |


## Takeaways

- SQL es el lenguaje base para convertir datos tabulares en evidencia analítica.
- En Colab podemos trabajar SQL sin instalar un motor local.
- DuckDB permite consultar tablas en memoria con sintaxis SQL.
- Antes de analizar, se validan tablas, conteos y llaves.
- `SELECT`, `WHERE`, `GROUP BY` y `JOIN` forman el núcleo de una consulta analítica.
- Los KPIs financieros deben tener fórmula, filtro, unidad e interpretación.
- Un buen analista no solo calcula: también comunica supuestos, hallazgos e implicaciones.


## Cierre

En esta sesión pasamos de archivos Parquet en GitHub a consultas SQL en Google Colab.

Flujo final:

```text
GitHub /datasets → Parquet → pandas → DuckDB → SQL → DataFrame → interpretación
```

Para la siguiente sesión, el foco será práctico: resolver preguntas de negocio con SQL, depurar errores y documentar resultados.

### Canales de apoyo

- `DATA-CO-LEARNING`: horarios de atención para dudas puntuales.
- `DA_CONSULTA`: preguntas sobre contenido o proyecto usando el tag correspondiente.
- `SPRINT FOCUS`: sesiones extra para profundizar en temas del sprint.
- `SESIONES 1:1`: tutorías individuales agendadas con antelación.
- Canal de Discord de la cohorte: espacio para compartir recursos, dudas y aprendizajes.

